# Pipeline 2 - GNN+GRU Flood Prediction

This notebook trains one GNN+GRU model from the existing Pipeline 2 graph datasets for Assam, Kerala, and Odisha. Each state graph is offset and combined into one disconnected graph, so no spatial edge is created between states.

It does not regenerate preprocessing outputs. It uses rainfall/river sequences in the GRU, combines the resulting temporal context with node features, and applies graph message passing to predict node-level flood probability. Training uses a 2023 held-out validation split when available.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import json
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit

import torch
from torch import nn

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


## 1. Paths and training settings

The input folders are created by `pipeline2_graph_dataset_preprocessing_colab.ipynb`. Change only `DRIVE_ROOT` if the project data is stored elsewhere.

In [ ]:
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/PRJ3_Data'),
    Path('/content/drive/MyDrive/Prj_3_Data'),
    Path('/content/drive/MyDrive/PRJ3_DATA'),
]
DRIVE_ROOT = next((p for p in DRIVE_CANDIDATES if p.exists()), DRIVE_CANDIDATES[0])
GRAPH_ROOT = DRIVE_ROOT / 'Pipeline2_Graph_Datasets'
OUTPUT_ROOT = DRIVE_ROOT / 'Pipeline2_GNN_GRU_Outputs'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
REPORT_DIR = OUTPUT_ROOT / 'reports'
PLOT_DIR = OUTPUT_ROOT / 'plots'
for directory in [OUTPUT_ROOT, CHECKPOINT_DIR, REPORT_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

STATES = ['Assam', 'Kerala', 'Odisha']
VAL_YEARS = [2023]
HIDDEN_DIM = 64
GRU_HIDDEN_DIM = 32
GRAPH_LAYERS = 2
DROPOUT = 0.20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 100
PATIENCE = 15
RESET_TRAINING = False
USE_AMP = torch.cuda.is_available()
BEST_CHECKPOINT = CHECKPOINT_DIR / 'best_checkpoint.pt'
LAST_CHECKPOINT = CHECKPOINT_DIR / 'last_checkpoint.pt'
HISTORY_PATH = REPORT_DIR / 'training_history.csv'
print('Graph root:', GRAPH_ROOT)
print('Output root:', OUTPUT_ROOT)
print('States:', STATES)


## 2. Load and combine disconnected state graphs

Node and sequence arrays are loaded state by state. Edge indices are offset by the current node count before concatenation. The resulting `edge_index` therefore contains spatial edges within each state only.

In [ ]:
def state_dir(state):
    return GRAPH_ROOT / state.lower().replace(' ', '_')

def load_graph_dataset(state):
    directory = state_dir(state)
    manifest_path = directory / 'manifest.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f'Missing Pipeline 2 manifest for {state}: {manifest_path}')
    manifest = json.loads(manifest_path.read_text())
    required = ['node_features.npy', 'edge_index.npy', 'target_fraction.npy',
                'target_binary.npy', 'node_metadata.csv', 'hydro_sequences.npy',
                'sequence_years.npy', 'node_sequence_index.npy']
    missing = [name for name in required if not (directory / name).exists()]
    if missing:
        raise FileNotFoundError(f'{state} is missing graph files: {missing}')
    features = np.load(directory / 'node_features.npy', mmap_mode='r')
    edges = np.load(directory / 'edge_index.npy', mmap_mode='r')
    target_fraction = np.load(directory / 'target_fraction.npy', mmap_mode='r')
    target_binary = np.load(directory / 'target_binary.npy', mmap_mode='r')
    sequences = np.load(directory / 'hydro_sequences.npy', mmap_mode='r')
    sequence_years = np.load(directory / 'sequence_years.npy', mmap_mode='r')
    node_sequence_index = np.load(directory / 'node_sequence_index.npy', mmap_mode='r')
    metadata = pd.read_csv(directory / 'node_metadata.csv')
    if edges.ndim != 2 or edges.shape[0] != 2 or edges.shape[1] == 0:
        raise ValueError(f'{state}: edge_index is empty or malformed. Re-run the updated Pipeline 2 preprocessing notebook for this state.')
    if len(metadata) != features.shape[0]:
        raise ValueError(f'{state}: metadata rows do not match node features.')
    if features.shape[0] != len(target_binary) or features.shape[0] != len(node_sequence_index):
        raise ValueError(f'{state}: node arrays have inconsistent lengths.')
    return {'state': state, 'manifest': manifest, 'features': np.asarray(features, dtype=np.float32),
            'edges': np.asarray(edges, dtype=np.int64), 'target_fraction': np.asarray(target_fraction, dtype=np.float32),
            'target_binary': np.asarray(target_binary, dtype=np.int64), 'sequences': np.asarray(sequences, dtype=np.float32),
            'sequence_years': np.asarray(sequence_years, dtype=np.int64),
            'node_sequence_index': np.asarray(node_sequence_index, dtype=np.int64), 'metadata': metadata}

state_datasets = [load_graph_dataset(state) for state in STATES]
feature_dim = state_datasets[0]['features'].shape[1]
sequence_feature_dim = state_datasets[0]['sequences'].shape[-1]
if any(item['features'].shape[1] != feature_dim for item in state_datasets):
    raise ValueError('All states must have the same node feature dimension.')
if any(item['sequences'].shape[-1] != sequence_feature_dim for item in state_datasets):
    raise ValueError('All states must have the same sequence feature dimension.')

feature_parts, target_fraction_parts, target_binary_parts = [], [], []
edge_parts, sequence_parts, sequence_index_parts, metadata_parts = [], [], [], []
node_offset = 0
sequence_offset = 0
state_ranges = []
for item in state_datasets:
    n = item['features'].shape[0]
    e = item['edges'].copy()
    if e.size:
        e += node_offset
    edge_parts.append(e)
    feature_parts.append(item['features'])
    target_fraction_parts.append(item['target_fraction'])
    target_binary_parts.append(item['target_binary'])
    sequence_parts.append(item['sequences'])
    sequence_index_parts.append(item['node_sequence_index'] + sequence_offset)
    metadata = item['metadata'].copy()
    metadata['state'] = item['state']
    metadata_parts.append(metadata)
    state_ranges.append({'state': item['state'], 'start': node_offset, 'end': node_offset + n,
                        'nodes': n, 'edges': int(e.shape[1])})
    node_offset += n
    sequence_offset += item['sequences'].shape[0]

pd.DataFrame(state_ranges).to_csv(REPORT_DIR / 'combined_graph_components.csv', index=False)
X = np.concatenate(feature_parts, axis=0).astype(np.float32, copy=False)
target_fraction = np.concatenate(target_fraction_parts, axis=0).astype(np.float32, copy=False)
target_binary = np.concatenate(target_binary_parts, axis=0).astype(np.int64, copy=False)
edge_index = np.concatenate(edge_parts, axis=1).astype(np.int64, copy=False)
hydro_sequences = np.concatenate(sequence_parts, axis=0).astype(np.float32, copy=False)
node_sequence_index = np.concatenate(sequence_index_parts, axis=0).astype(np.int64, copy=False)
metadata = pd.concat(metadata_parts, ignore_index=True)
metadata['year'] = pd.to_numeric(metadata['year'], errors='coerce').astype('Int64')
metadata['global_node_id'] = np.arange(len(metadata), dtype=np.int64)
if edge_index.size and (edge_index.min() < 0 or edge_index.max() >= len(X)):
    raise ValueError('Combined edge index contains an invalid node id.')

print('Combined nodes:', len(X))
print('Combined edges:', edge_index.shape[1])
print('Node feature dimension:', feature_dim)
print('Hydrological sequences:', hydro_sequences.shape)
display(pd.DataFrame(state_ranges))


## 3. Build the held-out validation split

The primary split holds out all 2023 nodes across the three states. If 2023 is unavailable or produces an unusable class split, the code falls back to a grouped state-year split.

In [ ]:
year_values = metadata['year'].to_numpy()
val_mask = np.isin(year_values, VAL_YEARS)
train_mask = ~val_mask
split_strategy = f'held_out_years_{VAL_YEARS}'
if (not val_mask.any() or not train_mask.any() or
        np.unique(target_binary[train_mask]).size < 2 or np.unique(target_binary[val_mask]).size < 2):
    groups = metadata['state'].astype(str).to_numpy() + '_' + metadata['year'].astype(str).to_numpy()
    train_indices, val_indices = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, target_binary, groups=groups))
    train_mask = np.zeros(len(X), dtype=bool)
    train_mask[train_indices] = True
    val_mask = ~train_mask
    split_strategy = 'grouped_state_year_fallback'
if np.unique(target_binary[train_mask]).size < 2 or np.unique(target_binary[val_mask]).size < 2:
    raise ValueError('Both training and validation splits must contain flood and non-flood nodes.')

split_summary = pd.DataFrame([
    {'split': 'train', 'nodes': int(train_mask.sum()), 'flood_rate': float(target_binary[train_mask].mean()), 'strategy': split_strategy},
    {'split': 'validation', 'nodes': int(val_mask.sum()), 'flood_rate': float(target_binary[val_mask].mean()), 'strategy': split_strategy},
])
split_summary.to_csv(REPORT_DIR / 'split_summary.csv', index=False)
metadata.assign(split=np.where(train_mask, 'train', 'validation')).groupby(['state', 'year', 'split'])['global_node_id'].agg(nodes='size').reset_index().to_csv(REPORT_DIR / 'state_year_split_summary.csv', index=False)
display(split_summary)


## 4. GNN+GRU model

The GRU encodes each rainfall/river sequence. Its year-level representation is assigned to the corresponding nodes through `node_sequence_index`. Two lightweight GraphSAGE-style layers then aggregate neighboring node features. No edge is added between states.

In [ ]:
class MeanGraphLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.self_linear = nn.Linear(in_features, out_features)
        self.neighbor_linear = nn.Linear(in_features, out_features)

    def forward(self, x, edge_index):
        source, destination = edge_index[0], edge_index[1]
        neighbor_sum = torch.zeros_like(x)
        neighbor_sum.index_add_(0, destination, x[source])
        degree = torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)
        degree.index_add_(0, destination, torch.ones(destination.shape[0], device=x.device, dtype=x.dtype))
        neighbor_mean = neighbor_sum / degree.clamp_min(1).unsqueeze(1)
        return self.self_linear(x) + self.neighbor_linear(neighbor_mean)

class GNNGRU(nn.Module):
    def __init__(self, node_dim, sequence_dim, gru_hidden=GRU_HIDDEN_DIM, hidden=HIDDEN_DIM, layers=GRAPH_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.gru = nn.GRU(sequence_dim, gru_hidden, batch_first=True)
        self.input = nn.Linear(node_dim + gru_hidden, hidden)
        self.graph_layers = nn.ModuleList([MeanGraphLayer(hidden, hidden) for _ in range(max(1, layers - 1))])
        self.output = nn.Linear(hidden, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, node_features, edge_index, sequences, node_sequence_index):
        sequence_embeddings, _ = self.gru(sequences)
        temporal_context = sequence_embeddings[node_sequence_index, -1, :]
        h = torch.cat([node_features, temporal_context], dim=1)
        h = torch.relu(self.input(h))
        for layer in self.graph_layers:
            h = torch.relu(layer(h, edge_index))
            h = self.dropout(h)
        return self.output(h).squeeze(1)

model = GNNGRU(feature_dim, sequence_feature_dim).to(DEVICE)
print(model)
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))


## 5. Resumable training

`last_checkpoint.pt` is written after every epoch and contains the model, optimizer, scheduler, scaler, epoch, best validation loss, and history. `best_checkpoint.pt` is updated whenever validation loss improves. Rerunning this section resumes automatically unless `RESET_TRAINING=True`.

In [ ]:
x_tensor = torch.from_numpy(X).to(DEVICE)
edge_tensor = torch.from_numpy(edge_index).to(DEVICE)
sequence_tensor = torch.from_numpy(hydro_sequences).to(DEVICE)
sequence_index_tensor = torch.from_numpy(node_sequence_index).to(DEVICE)
target_tensor = torch.from_numpy(target_binary.astype(np.float32)).to(DEVICE)
train_index_tensor = torch.from_numpy(np.flatnonzero(train_mask)).to(DEVICE)
val_index_tensor = torch.from_numpy(np.flatnonzero(val_mask)).to(DEVICE)

positive = float(target_binary[train_mask].sum())
negative = float(train_mask.sum() - positive)
pos_weight = torch.tensor([negative / max(positive, 1.0)], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
history = []
start_epoch = 1
best_val_loss = float('inf')
best_epoch = 0

signature = {
    'states': STATES, 'nodes': int(len(X)), 'feature_dim': int(feature_dim),
    'sequence_feature_dim': int(sequence_feature_dim), 'val_years': VAL_YEARS,
    'split_strategy': split_strategy, 'model': 'GNNGRU',
}

def save_checkpoint(path, epoch, best_loss):
    payload = {
        'epoch': int(epoch), 'best_val_loss': float(best_loss), 'best_epoch': int(best_epoch),
        'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(), 'scaler_state_dict': scaler.state_dict(),
        'history': history, 'signature': signature, 'seed': SEED,
    }
    torch.save(payload, path)

if LAST_CHECKPOINT.exists() and not RESET_TRAINING:
    checkpoint = torch.load(LAST_CHECKPOINT, map_location=DEVICE)
    if checkpoint.get('signature') != signature:
        raise ValueError('Existing checkpoint signature does not match the current graph dataset or split.')
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if checkpoint.get('scaler_state_dict'):
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    history = checkpoint.get('history', [])
    start_epoch = int(checkpoint['epoch']) + 1
    best_val_loss = float(checkpoint.get('best_val_loss', float('inf')))
    best_epoch = int(checkpoint.get('best_epoch', 0))
    print('Resumed from epoch', start_epoch - 1)
else:
    print('Starting a new training run.')

def run_epoch(training=True):
    model.train(training)
    index = train_index_tensor if training else val_index_tensor
    with torch.set_grad_enabled(training):
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(x_tensor, edge_tensor, sequence_tensor, sequence_index_tensor)
            loss = criterion(logits[index], target_tensor[index])
    if training:
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    return float(loss.detach().cpu())

for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = run_epoch(training=True)
    val_loss = run_epoch(training=False)
    scheduler.step(val_loss)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
           'learning_rate': optimizer.param_groups[0]['lr']}
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
    save_checkpoint(LAST_CHECKPOINT, epoch, best_val_loss)
    if improved:
        save_checkpoint(BEST_CHECKPOINT, epoch, best_val_loss)
    print(f'Epoch {epoch:03d} | train {train_loss:.5f} | val {val_loss:.5f} | lr {row["learning_rate"]:.2e}')
    if epoch - best_epoch >= PATIENCE:
        print('Early stopping.')
        break

print('Best checkpoint:', BEST_CHECKPOINT)
print('Last checkpoint:', LAST_CHECKPOINT)


## 6. Evaluate and save predictions

Evaluation is regenerated from the best checkpoint. Outputs include overall and per-state metrics, classification reports, node-level predictions, and training curves.

In [ ]:
if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError('Best checkpoint was not created. Run the training cell first.')
best_checkpoint = torch.load(BEST_CHECKPOINT, map_location=DEVICE)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()
with torch.inference_mode():
    logits = model(x_tensor, edge_tensor, sequence_tensor, sequence_index_tensor)
probability = torch.sigmoid(logits).cpu().numpy().astype(np.float32)
prediction = (probability >= 0.5).astype(np.uint8)

def metric_row(name, mask):
    y_true, y_prob, y_pred = target_binary[mask], probability[mask], prediction[mask]
    row = {'split': name, 'nodes': int(mask.sum()), 'flood_rate': float(y_true.mean()),
           'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0),
           'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0),
           'roc_auc': roc_auc_score(y_true, y_prob) if np.unique(y_true).size == 2 else np.nan,
           'average_precision': average_precision_score(y_true, y_prob) if np.unique(y_true).size == 2 else np.nan}
    return row

metrics = [metric_row('train', train_mask), metric_row('validation', val_mask)]
state_masks = {state: metadata['state'].eq(state).to_numpy() & val_mask for state in STATES}
metrics.extend(metric_row(f'validation_{state.lower()}', mask) for state, mask in state_masks.items() if mask.any())
metrics_table = pd.DataFrame(metrics)
metrics_table.to_csv(REPORT_DIR / 'evaluation_metrics.csv', index=False)
classification = classification_report(target_binary[val_mask], prediction[val_mask], target_names=['non_flood', 'flood'], zero_division=0)
(REPORT_DIR / 'validation_classification_report.txt').write_text(classification)

id_columns = [c for c in ['global_node_id', 'state', 'year', 'grid_row', 'grid_col', 'node_id', 'local_node_id', 'source_chunk'] if c in metadata.columns]
predictions = metadata[id_columns].copy()
predictions['target_fraction'] = target_fraction
predictions['target_binary'] = target_binary
predictions['flood_probability'] = probability
predictions['prediction'] = prediction
predictions['split'] = np.where(train_mask, 'train', 'validation')
predictions.to_csv(REPORT_DIR / 'node_predictions.csv', index=False)

history_frame = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_frame['epoch'], history_frame['train_loss'], label='train')
ax.plot(history_frame['epoch'], history_frame['val_loss'], label='validation')
ax.set(xlabel='Epoch', ylabel='BCE loss', title='GNN+GRU training history')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'training_history.png', dpi=160)
plt.show()

cm = confusion_matrix(target_binary[val_mask], prediction[val_mask])
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(cm, cmap='Blues')
fig.colorbar(image, ax=ax)
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['non_flood', 'flood'], yticklabels=['non_flood', 'flood'], xlabel='Predicted', ylabel='True', title='Validation confusion matrix')
for r in range(2):
    for c in range(2):
        ax.text(c, r, str(cm[r, c]), ha='center', va='center')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'validation_confusion_matrix.png', dpi=160)
plt.show()
display(metrics_table)
print('Predictions:', REPORT_DIR / 'node_predictions.csv')
print('Reports:', REPORT_DIR)


## 7. Saved outputs

```text
Pipeline2_GNN_GRU_Outputs/
  checkpoints/best_checkpoint.pt
  checkpoints/last_checkpoint.pt
  reports/training_history.csv
  reports/evaluation_metrics.csv
  reports/validation_classification_report.txt
  reports/node_predictions.csv
  reports/split_summary.csv
  plots/training_history.png
  plots/validation_confusion_matrix.png
```